# M03 — Curve Construction, Portfolio Valuation, and DV01

This notebook reproduces M03 from a fresh Google Colab runtime. It builds a continuously compounded zero curve, calibrates the 3Y/7Y/15Y synthetic Treasury portfolio, and checks frozen-position full repricing against linear DV01. It does not calculate VaR or Expected Shortfall.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/JoyWu-302121/market_risk.git'
PROJECT_DIR = Path('/content/market_risk') if IN_COLAB else Path.cwd().resolve()

if IN_COLAB and not (PROJECT_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_DIR)], check=True)
elif IN_COLAB:
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)

os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', 'requirements-colab.txt'], check=True)
src_path = str(PROJECT_DIR / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f'Project directory: {PROJECT_DIR}')
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)

## Load configuration and reproduce the accepted GSW data pipeline

In [ ]:
import pandas as pd
import yaml
from IPython.display import display

from bond_risk.data.pipeline import run_gsw_pipeline

data_configuration = yaml.safe_load((PROJECT_DIR / 'configs/data_sources.yaml').read_text())['gsw']
portfolio_configuration = yaml.safe_load((PROJECT_DIR / 'configs/portfolio.yaml').read_text())
OUTPUT_ROOT = PROJECT_DIR / 'data'

ingestion = run_gsw_pipeline(
    OUTPUT_ROOT,
    analysis_start=data_configuration['analysis_start'],
    required_tenors=data_configuration['required_tenors_years'],
    source_url=data_configuration['url'],
    stale_after_days=data_configuration['stale_after_calendar_days'],
)
assert ingestion['audit']['status'] != 'FAIL', ingestion['audit']['failures']
curve_data = pd.read_csv(ingestion['processed_path'], parse_dates=['observation_date'])
ingestion['audit']

## Construct the latest zero curve

The curve stores decimal continuously compounded zero rates and interpolates linearly in log discount factors. Extrapolation outside the observed curve is rejected.

In [ ]:
from bond_risk.curves import ZeroCurve

curve = ZeroCurve.from_long_frame(curve_data)
curve_nodes = curve.to_frame()
display(curve_nodes.head())
print(f'Curve date: {curve.observation_date}')
print(f'Curve range: {curve.minimum_tenor:.0f}Y to {curve.maximum_tenor:.0f}Y')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(curve_nodes['tenor_years'], 100 * curve_nodes['zero_rate_cc'], marker='o')
axes[0].set(title=f'GSW Zero Curve — {curve.observation_date}', xlabel='Maturity (years)', ylabel='Zero yield (%)')
axes[0].grid(alpha=0.3)
axes[1].plot(curve_nodes['tenor_years'], curve_nodes['discount_factor'], marker='o')
axes[1].set(title='Discount Factors', xlabel='Maturity (years)', ylabel='Discount factor')
axes[1].grid(alpha=0.3)
plt.tight_layout()

## Calibrate and value the phase-one portfolio

Quantities are chosen so the three positions each represent one-third of the USD 10 million market value on the curve date.

In [ ]:
from bond_risk.portfolio import build_target_weight_portfolio

portfolio = build_target_weight_portfolio(
    portfolio_id=portfolio_configuration['portfolio_id'],
    curve=curve,
    total_market_value=portfolio_configuration['initial_market_value'],
    instrument_specs=portfolio_configuration['positions'],
)
bump = portfolio_configuration['dv01']['parallel_bump_decimal']
position_report = portfolio.position_report(curve, bump)
display(position_report)

## Validate full repricing and DV01

In [ ]:
base_value = portfolio.value(curve)
zero_shock_pnl = portfolio.pnl(curve, curve.parallel_shift(0.0))
full_dv01 = portfolio.full_revaluation_parallel_dv01(curve, bump)
linear_dv01 = portfolio.linear_parallel_dv01(curve, bump)
relative_error = abs(linear_dv01 - full_dv01) / full_dv01
tolerance = portfolio_configuration['dv01']['maximum_relative_linearization_error']

assert abs(base_value - portfolio_configuration['initial_market_value']) < 1e-6
assert abs(zero_shock_pnl) < 1e-12
assert full_dv01 > 0
assert relative_error <= tolerance
assert (abs(position_report['actual_weight'] - position_report['target_weight']) <= 1e-12).all()

print(f'Base market value: USD {base_value:,.2f}')
print(f'Full-revaluation DV01: USD {full_dv01:,.2f}')
print(f'Linear DV01: USD {linear_dv01:,.2f}')
print(f'Linearization relative error: {relative_error:.4%}')
print('M03 acceptance checks: PASS')

## M03 completion boundary

M03 validates a single-date curve and frozen-position valuation engine. M04 will apply historical curve shocks to this same portfolio and valuation function to calculate historical-simulation VaR and Expected Shortfall.